# **ИКБО-62-23 Чекалов А.В. 07.10.2026**

# **1. Учебный блок по ecommerce.sqlite**

Задаются переменные окружения JAVA_HOME и PATH, чтобы Spark мог найти установленную Java 17. 
Без этого PySpark не запустится.

In [1]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]

Выводятся версии Python, Java и PySpark. 
Это нужно, чтобы зафиксировать, в каком окружении выполнялась работа.

In [2]:
import sys
import subprocess
import pyspark

print("Python:", sys.version, end = "\n\n")

result = subprocess.run(['java', '-version'], capture_output=True, text=True)
print(result.stderr)

print("PySpark:", pyspark.__version__)

Python: 3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]

java version "17.0.12" 2024-07-16 LTS
Java(TM) SE Runtime Environment (build 17.0.12+8-LTS-286)
Java HotSpot(TM) 64-Bit Server VM (build 17.0.12+8-LTS-286, mixed mode, sharing)

PySpark: 4.2.0


Открывается база ecommerce.sqlite, из неё выгружаются 7 таблиц (orders, order_items, products, customers, order_payments, order_reviews, перевод категорий) в отдельные CSV-файлы в папке spark_input. 
Для каждой таблицы печатается число выгруженных строк.

In [3]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("ecommerce.sqlite")
STAGING = Path("spark_input")
STAGING.mkdir(exist_ok=True)

TABLES = [
    "orders",
    "order_items",
    "products",
    "customers",
    "order_payments",
    "order_reviews",
    "product_category_name_translation",
]

with sqlite3.connect(DB_PATH) as conn:
    for table in TABLES:
        part = pd.read_sql_query(
            f'SELECT * FROM "{table}"',
            conn
        )
        part.to_csv(
            STAGING / f"{table}.csv",
            index=False,
            encoding="utf-8"
        )
        print(f"{table:36s} {len(part):>8} rows")

orders                                  99441 rows
order_items                            112650 rows
products                                32951 rows
customers                               99441 rows
order_payments                         103886 rows
order_reviews                           97621 rows
product_category_name_translation          71 rows


Задаются переменные окружения Hadoop, 
затем создаётся локальная SparkSession с именем Practice4_Ecommerce, 
режимом local[*] (Spark запускается не на кластере, а на одном компьютере и использует все доступные логические ядра этого компьютера), 
памятью драйвера 2 ГБ и 8 разделами для shuffle. 
Выводятся master, имя приложения и версия Spark.

In [4]:
import os
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Practice4_Ecommerce")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Master:", spark.sparkContext.master)
print("App:", spark.sparkContext.appName)
print("Spark:", spark.version)

C:\Users\lesha\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Master: local[*]
App: Practice4_Ecommerce
Spark: 4.2.0


CSV с заказами читается как есть, без указания схемы — все поля остаются строками (string). 
Печатается схема, число строк и первые 5 записей.

In [5]:
orders_raw = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .csv(str(STAGING / "orders.csv"))
)

orders_raw.printSchema()
print("Строк:", orders_raw.count())

orders_raw.select(
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp"
).show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)

Строк: 99441
+--------------------------------+--------------------------------+------------+------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|
+--------------------------------+--------------------------------+------------+------------------------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|delivered   |2017-10-02 10:56:33     |
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|delivered   |2018-07-24 20:41:37     |
|47770eb9100c2d0c44946d9cf07ec65d|41c

Для таблицы позиций заказов задаётся StructType: price и freight_value сразу читаются как числа (double), order_item_id — как целое. 
Печатается схема и число строк.

In [6]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

order_items_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_item_id", IntegerType(), True),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])

items = (
    spark.read
    .option("header", True)
    .schema(order_items_schema)
    .csv(str(STAGING / "order_items.csv"))
)

items.printSchema()
print("Строк:", items.count())

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

Строк: 112650


Пять строковых столбцов с датами заказов приводятся к типу timestamp. 
После этого с ними можно выполнять операции с датами (разница, формат, сравнение).

In [7]:
timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

orders = orders_raw
for c in timestamp_cols:
    orders = orders.withColumn(
        c,
        F.to_timestamp(F.col(c), "yyyy-MM-dd HH:mm:ss")
    )

orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



Из позиций заказов выбираются те, где price >= 500, добавляется вычисляемый признак gross_item_value (цена + доставка). 
Печатается число таких позиций и топ-5 по величине.

In [8]:
expensive_items = (
    items
    .filter(F.col("price") >= 500)
    .withColumn(
        "gross_item_value",
        F.round(F.col("price") + F.col("freight_value"), 2)
    )
    .select(
        "order_id", "order_item_id",
        "price", "freight_value", "gross_item_value"
    )
)

print("Позиций с price >= 500:", expensive_items.count())

expensive_items.orderBy(
    F.desc("gross_item_value")
).show(5, truncate=False)

Позиций с price >= 500: 3239
+--------------------------------+-------------+------+-------------+----------------+
|order_id                        |order_item_id|price |freight_value|gross_item_value|
+--------------------------------+-------------+------+-------------+----------------+
|0812eb902a67711a1cb742b3cdaa65ae|1            |6735.0|194.31       |6929.31         |
|fefacc66af859508bf1a7934eab1e97f|1            |6729.0|193.21       |6922.21         |
|f5136e38d1a14a4dbd87dff67da82701|1            |6499.0|227.66       |6726.66         |
|a96610ab360d42a2e5335a3998b4718a|1            |4799.0|151.34       |4950.34         |
|199af31afc78c699f0dbf71fb178d4d4|1            |4690.0|74.34        |4764.34         |
+--------------------------------+-------------+------+-------------+----------------+
only showing top 5 rows


Из orders берутся только доставленные заказы, добавляются: месяц покупки, число дней доставки и флаг просрочки. 
Печатается: сколько доставлено, у скольких есть фактическая дата доставки, у скольких была просрочка.

In [9]:
delivered = (
    orders
    .filter(F.col("order_status") == "delivered")
    .withColumn(
        "purchase_month",
        F.date_format("order_purchase_timestamp", "yyyy-MM")
    )
    .withColumn(
        "delivery_days",
        F.datediff(
            "order_delivered_customer_date",
            "order_purchase_timestamp"
        )
    )
    .withColumn(
        "is_late",
        F.col("order_delivered_customer_date") >
        F.col("order_estimated_delivery_date")
    )
)

with_delivery_date = delivered.filter(
    F.col("order_delivered_customer_date").isNotNull()
)
late = with_delivery_date.filter(F.col("is_late"))

print("Доставленных заказов:", delivered.count()) 
print("С известной датой доставки:", with_delivery_date.count()) 
print("Позже ожидаемой даты:", late.count())

Доставленных заказов: 96478
С известной датой доставки: 96470
Позже ожидаемой даты: 7826


Для каждого из пяти временных столбцов заказов считается число null. 
Видно, что purchase и estimated delivery заполнены полностью, а в delivered_customer_date — 2 965 пропусков.

In [10]:
null_counts = orders.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in timestamp_cols
])

null_counts.show(vertical=True)

-RECORD 0-----------------------------
 order_purchase_timestamp      | 0    
 order_approved_at             | 160  
 order_delivered_carrier_date  | 1783 
 order_delivered_customer_date | 2965 
 order_estimated_delivery_date | 0    



Отбираются заказы без фактической даты доставки, группируются по статусу. 
Видно, что основная масса пропусков приходится на shipped, canceled, unavailable — то есть на незавершённые процессы, а не на случайную потерю данных.

In [11]:
missing_delivery_by_status = (
    orders
    .filter(F.col("order_delivered_customer_date").isNull())
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

missing_delivery_by_status.show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|shipped     |1107 |
|canceled    |619  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|delivered   |8    |
|created     |5    |
|approved    |2    |
+------------+-----+



Читается таблица products. 
Считается число товаров без категории и пропуски в категории заполняются значением unknown, чтобы не терять эти товары при JOIN.

In [12]:
products = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(STAGING / "products.csv"))
)

missing_categories = products.filter(
    F.col("product_category_name").isNull()
).count()

products_clean = products.fillna({
    "product_category_name": "unknown"
})

print("Товаров без категории:", missing_categories)

Товаров без категории: 610


Группировка orders по order_status с подсчётом числа заказов. 
Видно, что почти все заказы в статусе delivered.

In [13]:
status_stats = (
    orders
    .groupBy("order_status")
    .count()
    .orderBy(F.desc("count"))
)

status_stats.show(truncate=False)

+------------+-----+
|order_status|count|
+------------+-----+
|delivered   |96478|
|shipped     |1107 |
|canceled    |625  |
|unavailable |609  |
|invoiced    |314  |
|processing  |301  |
|created     |5    |
|approved    |2    |
+------------+-----+



Читается таблица order_payments. 
По каждому типу платежа считаются: количество платежей, суммарная стоимость и средний платёж. 
Доминирует credit_card.

In [14]:
payments = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(STAGING / "order_payments.csv"))
)

payment_stats = (
    payments
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("payments"),
        F.round(F.sum("payment_value"), 2).alias("total_value"),
        F.round(F.avg("payment_value"), 2).alias("avg_value")
    )
    .orderBy(F.desc("total_value"))
)

payment_stats.show(truncate=False)

+------------+--------+-------------+---------+
|payment_type|payments|total_value  |avg_value|
+------------+--------+-------------+---------+
|credit_card |76795   |1.254208419E7|163.32   |
|boleto      |19784   |2869361.27   |145.03   |
|voucher     |5775    |379436.87    |65.7     |
|debit_card  |1529    |217989.79    |142.57   |
|not_defined |3       |0.0          |0.0      |
+------------+--------+-------------+---------+



К позициям заказов через LEFT JOIN присоединяются товары и справочник перевода категорий. 
Итоговая витрина sales сохраняет все 112 650 позиций; для 1 603 из них категория осталась unknown. 
Контроль числа строк до/после подтверждает корректность JOIN.

In [15]:
translations = (
    spark.read
    .option("header", True)
    .csv(str(STAGING / "product_category_name_translation.csv"))
)

sales = (
    items.alias("i")
    .join(
        products_clean.select(
            "product_id", "product_category_name"
        ).alias("p"),
        F.col("i.product_id") == F.col("p.product_id"),
        "left"
    )
    .join(
        translations.alias("t"),
        F.col("p.product_category_name") ==
        F.col("t.product_category_name"),
        "left"
    )
    .select(
        F.col("i.order_id").alias("order_id"),
        F.col("i.order_item_id").alias("order_item_id"),
        F.col("i.product_id").alias("product_id"),
        F.col("i.price").alias("price"),
        F.col("i.freight_value").alias("freight_value"),
        F.coalesce(
            F.col("t.product_category_name_english"),
            F.col("p.product_category_name"),
            F.lit("unknown")
        ).alias("category")
    )
)

print("Строк до JOIN:", items.count())
print("Строк после JOIN:", sales.count())
print(
    "Позиций категории unknown:",
    sales.filter(F.col("category") == "unknown").count()
)

Строк до JOIN: 112650
Строк после JOIN: 112650
Позиций категории unknown: 1603


К доставленным заказам присоединяется таблица customers, считается число доставок и просрочек по каждому штату, а также процент просрочки. 
Топ-1 по доле просрочек — AL (23,93 %), но на маленькой выборке.

In [16]:
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(STAGING / "customers.csv"))
)

late_by_state = (
    delivered
    .join(
        customers.select("customer_id", "customer_state"),
        on="customer_id",
        how="left"
    )
    .filter(F.col("order_delivered_customer_date").isNotNull())
    .groupBy("customer_state")
    .agg(
        F.count("*").alias("delivered"),
        F.sum(
            F.when(F.col("is_late"), 1).otherwise(0)
        ).alias("late")
    )
    .withColumn(
        "late_pct",
        F.round(F.col("late") / F.col("delivered") * 100, 2)
    )
    .orderBy(F.desc("late_pct"))
)

late_by_state.show(10, truncate=False)

+--------------+---------+----+--------+
|customer_state|delivered|late|late_pct|
+--------------+---------+----+--------+
|AL            |397      |95  |23.93   |
|MA            |717      |141 |19.67   |
|PI            |476      |76  |15.97   |
|CE            |1279     |196 |15.32   |
|SE            |335      |51  |15.22   |
|BA            |3256     |457 |14.04   |
|RJ            |12350    |1664|13.47   |
|TO            |274      |35  |12.77   |
|PA            |946      |117 |12.37   |
|ES            |1995     |244 |12.23   |
+--------------+---------+----+--------+
only showing top 10 rows


Витрина sales регистрируется как временное представление, через spark.sql считается выручка, число позиций, уникальных заказов и средняя цена по категориям. Топ-1 — health_beauty.

In [17]:
sales.createOrReplaceTempView("sales")

category_sql = spark.sql("""
    SELECT category,
           COUNT(*) AS items,
           COUNT(DISTINCT order_id) AS orders,
           ROUND(SUM(price), 2) AS revenue,
           ROUND(AVG(price), 2) AS avg_price
    FROM sales
    GROUP BY category
    ORDER BY revenue DESC
    LIMIT 10
""")

category_sql.show(truncate=False)

+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+



Эквивалентный запрос формируется через groupBy().agg(). 
Результат совпадает с SQL-версией — это подтверждает, что DataFrame API и Spark SQL работают через один оптимизатор. 
В витрине 74 категории.

In [18]:
category_sales = (
    sales
    .groupBy("category")
    .agg(
        F.count("*").alias("items"),
        F.countDistinct("order_id").alias("orders"),
        F.round(F.sum("price"), 2).alias("revenue"),
        F.round(F.avg("price"), 2).alias("avg_price")
    )
)

print("Категорий в витрине:", category_sales.count())

category_sales.orderBy(
    F.desc("revenue")
).show(10, truncate=False)

Категорий в витрине: 74
+---------------------+-----+------+----------+---------+
|category             |items|orders|revenue   |avg_price|
+---------------------+-----+------+----------+---------+
|health_beauty        |9670 |8836  |1258681.34|130.16   |
|watches_gifts        |5991 |5624  |1205005.68|201.14   |
|bed_bath_table       |11115|9417  |1036988.68|93.3     |
|sports_leisure       |8641 |7720  |988048.97 |114.34   |
|computers_accessories|7827 |6689  |911954.32 |116.51   |
|furniture_decor      |8334 |6449  |729762.49 |87.56    |
|cool_stuff           |3796 |3632  |635290.85 |167.36   |
|housewares           |6964 |5884  |632248.66 |90.79    |
|auto                 |4235 |3897  |592720.11 |139.96   |
|garden_tools         |4347 |3518  |485256.46 |111.63   |
+---------------------+-----+------+----------+---------+
only showing top 10 rows


Итоговая таблица по категориям записывается в Parquet (с сохранением схемы) и в CSV (один part-файл через coalesce(1)). 
Затем Parquet читается обратно — 74 строки и схема сохранились.

In [19]:
OUT = Path("spark_output")
OUT.mkdir(exist_ok=True)

(
    category_sales
    .write
    .mode("overwrite")
    .parquet(str(OUT / "category_sales_parquet"))
)

(
    category_sales
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(str(OUT / "category_sales_csv"))
)

category_back = spark.read.parquet(
    str(OUT / "category_sales_parquet")
)
print("Строк после round-trip:", category_back.count())
category_back.printSchema()

Строк после round-trip: 74
root
 |-- category: string (nullable = true)
 |-- items: long (nullable = true)
 |-- orders: long (nullable = true)
 |-- revenue: double (nullable = true)
 |-- avg_price: double (nullable = true)



Освобождаются ресурсы локального Spark. Это последняя ячейка, после неё никакие DataFrame из этой сессии использовать нельзя.

In [20]:
spark.stop()

### **Выводы по блоку 1**
1. Заказы в основном завершены — 97 % имеют статус delivered, остальные статусы редки.

2. Пропуски даты доставки отражают состояние процесса, а не ошибку данных — почти все они у незавершённых или отменённых заказов.

3. Просрочка составляет ≈8,11 % доставок; максимум по доле — AL (23,93 %), но на малой выборке (397 заказов).

4. Выручка сконцентрирована в нескольких категориях: health_beauty, watches_gifts, bed_bath_table.

5. Оплата почти полностью карточная — credit_card покрывает 76 795 платежей из ~103 тыс.

# **2. Самостоятельный раздел по <br>2020_Мониторинг распространения ВИЧ-инфекции по субъектам РФ**

In [21]:
from pathlib import Path

DATA = Path("data.csv")
print("Файл существует:", DATA.exists())
print("Размер, байт:", DATA.stat().st_size)

Файл существует: True
Размер, байт: 88023


Файл data.csv найден, размер 88 023 байта. Staging не потребовался: Spark читает CSV напрямую, разделитель и кодировка стандартные — это упрощает pipeline и не создаёт лишних промежуточных файлов.

In [22]:
import os
import sys
import subprocess
import pyspark

os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-17"
os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Practice4_HIV")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.python.worker.timeout", "600")
    .config("spark.network.timeout", "600s")
    .config("spark.sql.broadcastTimeout", "600")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print("Python:", sys.version)
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)
print("PySpark:", pyspark.__version__)
print("Master:", spark.sparkContext.master)
print("App:", spark.sparkContext.appName)
print("Spark:", spark.version)

Python: 3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]
java version "17.0.12" 2024-07-16 LTS
Java(TM) SE Runtime Environment (build 17.0.12+8-LTS-286)
Java HotSpot(TM) 64-Bit Server VM (build 17.0.12+8-LTS-286, mixed mode, sharing)

PySpark: 4.2.0
Master: local[*]
App: Practice4_HIV
Spark: 4.2.0


Локальная сессия local[*] создана: Python 3.12.8, Java 17.0.12, PySpark 4.2.0, приложение Practice4_HIV. Дополнительные таймауты (spark.python.worker.timeout, spark.network.timeout) выставлены заранее — с ними длинные агрегации не падают по таймауту.

In [23]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")
    .csv(str(DATA))
)

print("Строк:", df.count())
print("Столбцов:", len(df.columns))
print(df.columns)
df.printSchema()

df.select(
    "territory", "year",
    "hiv_firstdiagnosis_100k",
    "hiv_regnum_firstdiagnosis",
    "hiv_russian_ibtestedpositive",
    "num_infectiousdoctors_inmedorgservices"
).show(5, truncate=False)

Строк: 760
Столбцов: 16
['territory', 'hiv_therapy_regdispensaries', 'hiv_regdispensaries_numdetect', 'hiv_firstdiagnosis_100k', 'hiv_motherchild_chemoprophylaxis', 'hiv_regnum_firstdiagnosis', 'hiv_totalnumrussian_tested', 'hiv_russian_ibtestedpositive', 'hiv_russian_ifatestedpositive', 'hiv_totalnumforeign_tested', 'hiv_foreign_ibtestedpositive', 'hiv_foreign_ifatestedpositive', 'num_alldoctors_inmedorgservices', 'num_infectiousdoctors_inmedorgservices', 'per_infectious_inalldoctors', 'year']
root
 |-- territory: string (nullable = true)
 |-- hiv_therapy_regdispensaries: double (nullable = true)
 |-- hiv_regdispensaries_numdetect: double (nullable = true)
 |-- hiv_firstdiagnosis_100k: double (nullable = true)
 |-- hiv_motherchild_chemoprophylaxis: double (nullable = true)
 |-- hiv_regnum_firstdiagnosis: double (nullable = true)
 |-- hiv_totalnumrussian_tested: double (nullable = true)
 |-- hiv_russian_ibtestedpositive: double (nullable = true)
 |-- hiv_russian_ifatestedpositive: doub

Датасет прочитан: 760 строк, 16 столбцов. inferSchema корректно распознала числовые поля как double, а year — как int. Уже на этом этапе видно, что в 2016–2018 часть тестовых полей пуста — это не ошибка, а смена методологии учёта.

In [24]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

schema = StructType([
    StructField("territory", StringType(), True),
    StructField("hiv_therapy_regdispensaries", DoubleType(), True),
    StructField("hiv_regdispensaries_numdetect", DoubleType(), True),
    StructField("hiv_firstdiagnosis_100k", DoubleType(), True),
    StructField("hiv_motherchild_chemoprophylaxis", DoubleType(), True),
    StructField("hiv_regnum_firstdiagnosis", DoubleType(), True),
    StructField("hiv_totalnumrussian_tested", DoubleType(), True),
    StructField("hiv_russian_ibtestedpositive", DoubleType(), True),
    StructField("hiv_russian_ifatestedpositive", DoubleType(), True),
    StructField("hiv_totalnumforeign_tested", DoubleType(), True),
    StructField("hiv_foreign_ibtestedpositive", DoubleType(), True),
    StructField("hiv_foreign_ifatestedpositive", DoubleType(), True),
    StructField("num_alldoctors_inmedorgservices", DoubleType(), True),
    StructField("num_infectiousdoctors_inmedorgservices", DoubleType(), True),
    StructField("per_infectious_inalldoctors", DoubleType(), True),
    StructField("year", IntegerType(), True),
])

df_s = (
    spark.read
    .option("header", True)
    .schema(schema)
    .csv(str(DATA))
)

df_s.printSchema()

df_str = spark.read.option("header", True).csv(str(DATA))

print("Строк:", df_s.count(), end = "\n\n")
print("Явная схема даёт:", df_s.dtypes[:5], end = "\n\n")
print("inferSchema дала :", df.dtypes[:5], end = "\n\n")
print("чтение строками дало :", df_str.dtypes[:5], end = "\n\n")

root
 |-- territory: string (nullable = true)
 |-- hiv_therapy_regdispensaries: double (nullable = true)
 |-- hiv_regdispensaries_numdetect: double (nullable = true)
 |-- hiv_firstdiagnosis_100k: double (nullable = true)
 |-- hiv_motherchild_chemoprophylaxis: double (nullable = true)
 |-- hiv_regnum_firstdiagnosis: double (nullable = true)
 |-- hiv_totalnumrussian_tested: double (nullable = true)
 |-- hiv_russian_ibtestedpositive: double (nullable = true)
 |-- hiv_russian_ifatestedpositive: double (nullable = true)
 |-- hiv_totalnumforeign_tested: double (nullable = true)
 |-- hiv_foreign_ibtestedpositive: double (nullable = true)
 |-- hiv_foreign_ifatestedpositive: double (nullable = true)
 |-- num_alldoctors_inmedorgservices: double (nullable = true)
 |-- num_infectiousdoctors_inmedorgservices: double (nullable = true)
 |-- per_infectious_inalldoctors: double (nullable = true)
 |-- year: integer (nullable = true)

Строк: 760

Явная схема даёт: [('territory', 'string'), ('hiv_therapy_

С StructType типы зафиксированы заранее и совпадают с тем, что дал inferSchema. А вот чтение строками (df_str) дало все числовые поля как string — значит, явная схема избавляет от повторного анализа файла и от риска, что Spark угадает тип неверно.

In [25]:
df_t = (
    df_s
    .filter(F.col("year") >= 2015)
    .filter(F.col("hiv_regnum_firstdiagnosis").isNotNull())
    .withColumn(
        "positivity_ru_pct",
        F.round(F.col("hiv_russian_ibtestedpositive") /
                F.col("hiv_totalnumrussian_tested") * 100, 3)
    )
    .withColumn(
        "positivity_foreign_pct",
        F.round(F.col("hiv_foreign_ibtestedpositive") /
                F.col("hiv_totalnumforeign_tested") * 100, 3)
    )
    .withColumn(
        "firstdiag_per_infdoctor",
        F.round(F.col("hiv_regnum_firstdiagnosis") /
                F.col("num_infectiousdoctors_inmedorgservices"), 1)
    )
    .select(
        "territory", "year",
        "hiv_regnum_firstdiagnosis",
        "positivity_ru_pct", "positivity_foreign_pct",
        "firstdiag_per_infdoctor"
    )
    .dropDuplicates(["territory", "year"])
    .orderBy(F.desc("hiv_regnum_firstdiagnosis"))
)

print("Строк после трансформаций:", df_t.count())
df_t.show(10, truncate=False)

Строк после трансформаций: 466
+-----------------------------+----+-------------------------+-----------------+----------------------+-----------------------+
|territory                    |year|hiv_regnum_firstdiagnosis|positivity_ru_pct|positivity_foreign_pct|firstdiag_per_infdoctor|
+-----------------------------+----+-------------------------+-----------------+----------------------+-----------------------+
|Российская Федерация         |2015|100220.0                 |0.428            |0.194                 |15.1                   |
|Российская Федерация         |2016|86855.0                  |NULL             |NULL                  |13.0                   |
|Российская Федерация         |2018|85995.0                  |NULL             |NULL                  |12.5                   |
|Российская Федерация         |2017|85802.0                  |NULL             |NULL                  |12.4                   |
|Российская Федерация         |2019|80124.0                  |0.294      

После фильтров и вычисляемых признаков (доля положительных среди россиян и иностранцев, число первичных диагнозов на одного инфекциониста) осталось 466 строк. Видно, что вычисляемые показатели теряют значения в 2016–2018 — это прямое следствие пропусков в исходных полях.

In [26]:
check_cols = [
    "hiv_therapy_regdispensaries",
    "hiv_regdispensaries_numdetect",
    "hiv_totalnumrussian_tested",
    "hiv_russian_ibtestedpositive",
    "hiv_regnum_firstdiagnosis",
]

df_s.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in check_cols
]).show(vertical=True)

df_clean = df_s.fillna({
    "hiv_therapy_regdispensaries": 0.0,
    "hiv_regdispensaries_numdetect": 0.0,
})

print("Строк до fillna:", df_s.count())
print("Строк после fillna:", df_clean.count())

-RECORD 0----------------------------
 hiv_therapy_regdispensaries   | 18  
 hiv_regdispensaries_numdetect | 19  
 hiv_totalnumrussian_tested    | 295 
 hiv_russian_ibtestedpositive  | 295 
 hiv_regnum_firstdiagnosis     | 16  

Строк до fillna: 760
Строк после fillna: 760


Пропуски распределены неравномерно: 295 в тестовых полях, 16–19 в показателях диспансеров. Тестовые поля не заполняем — это методологический разрыв, а не ошибка. Заполнение нулями только двух полей (hiv_therapy_regdispensaries, hiv_regdispensaries_numdetect) не изменило число строк и не исказило средние.

In [27]:
by_year = (
    df_clean
    .groupBy("year")
    .agg(
        F.count("*").alias("n_territories"),
        F.round(F.avg("hiv_firstdiagnosis_100k"), 2).alias("avg_firstdiag_100k"),
        F.sum("hiv_regnum_firstdiagnosis").alias("total_firstdiag"),
        F.round(F.avg("per_infectious_inalldoctors"), 2).alias("avg_share_inf")
    )
    .orderBy("year")
)
by_year.show(truncate=False)

by_terr = (
    df_clean
    .filter(F.col("year") == 2019)
    .groupBy("territory")
    .agg(
        F.max("hiv_firstdiagnosis_100k").alias("max_100k"),
        F.min("hiv_firstdiagnosis_100k").alias("min_100k"),
        F.avg("hiv_firstdiagnosis_100k").alias("avg_100k")
    )
    .orderBy(F.desc("max_100k"))
)
by_terr.show(10, truncate=False)

+----+-------------+------------------+---------------+-------------+
|year|n_territories|avg_firstdiag_100k|total_firstdiag|avg_share_inf|
+----+-------------+------------------+---------------+-------------+
|2012|95           |40.89             |228888.0       |1.16         |
|2013|95           |45.45             |245094.0       |1.15         |
|2014|95           |51.09             |277710.0       |1.08         |
|2015|95           |55.84             |300444.0       |1.1          |
|2016|95           |51.75             |260293.0       |1.08         |
|2017|95           |53.31             |257406.0       |1.09         |
|2018|95           |52.27             |256190.0       |1.09         |
|2019|95           |49.41             |238681.0       |1.09         |
+----+-------------+------------------+---------------+-------------+

+---------------------------+--------+--------+--------+
|territory                  |max_100k|min_100k|avg_100k|
+---------------------------+--------+-------

По годам: пик среднего первичного диагноза на 100 тыс. — в 2015 (55,84), затем снижение до 49,41 в 2019. По территориям за 2019 год максимум у Кемеровской области (164,3 на 100 тыс.). Число территорий в каждой группе стабильно — 95.

In [28]:
ref = spark.createDataFrame([
    ("Российская Федерация", "national"),
    ("г. Москва", "city"),
    ("г. Санкт-Петербург", "city"),
    ("г. Севастополь", "city"),
    ("Республика Крым", "region"),
    ("Центральный федеральный округ", "district"),
], ["territory", "level"])

left = df_clean.join(ref, on="territory", how="left")
inner = df_clean.join(ref, on="territory", how="inner")

print("До JOIN:", df_clean.count())
print("После LEFT JOIN:", left.count())
print("После INNER JOIN:", inner.count())
left.select("territory", "year", "level").show(5, truncate=False)

До JOIN: 760
После LEFT JOIN: 760
После INNER JOIN: 48
+--------------------+----+--------+
|territory           |year|level   |
+--------------------+----+--------+
|Российская Федерация|2012|national|
|Российская Федерация|2013|national|
|Российская Федерация|2014|national|
|Российская Федерация|2015|national|
|Российская Федерация|2016|national|
+--------------------+----+--------+
only showing top 5 rows


LEFT JOIN сохранил все 760 строк, INNER — только 48 (только территории, попавшие в справочник). Это подтверждает: для аналитики фактов безопаснее LEFT, иначе теряются регионы без справочной записи.

In [29]:
df_clean.createOrReplaceTempView("hiv")

spark.sql("""
    SELECT year,
           COUNT(*) AS n_territories,
           ROUND(AVG(hiv_firstdiagnosis_100k), 2) AS avg_100k
    FROM hiv
    GROUP BY year
    ORDER BY avg_100k DESC
""").show()

spark.sql("""
    SELECT territory, year, hiv_firstdiagnosis_100k
    FROM hiv
    WHERE hiv_firstdiagnosis_100k IS NOT NULL
    ORDER BY hiv_firstdiagnosis_100k DESC
    LIMIT 10
""").show(truncate=False)

+----+-------------+--------+
|year|n_territories|avg_100k|
+----+-------------+--------+
|2015|           95|   55.84|
|2017|           95|   53.31|
|2018|           95|   52.27|
|2016|           95|   51.75|
|2014|           95|   51.09|
|2019|           95|   49.41|
|2013|           95|   45.45|
|2012|           95|   40.89|
+----+-------------+--------+

+--------------------+----+-----------------------+
|territory           |year|hiv_firstdiagnosis_100k|
+--------------------+----+-----------------------+
|Кемеровская область |2014|239.8                  |
|Кемеровская область |2015|237.5                  |
|Кемеровская область |2012|219.4                  |
|Кемеровская область |2013|217.1                  |
|Кемеровская область |2016|201.8                  |
|Иркутская область   |2018|200.9                  |
|Томская область     |2013|173.0                  |
|Иркутская область   |2017|172.3                  |
|Кемеровская область |2017|170.2                  |
|Свердловская о

Через spark.sql получены те же результаты, что и через DataFrame API: топ по годам — 2015, топ территорий — Кемеровская область. Один оптимизатор, два интерфейса — результат совпадает.

In [30]:
OUT = Path("spark_output")
OUT.mkdir(exist_ok=True)

final = (
    df_clean
    .groupBy("year")
    .agg(
        F.count("*").alias("n_territories"),
        F.round(F.avg("hiv_firstdiagnosis_100k"), 2).alias("avg_firstdiag_100k"),
        F.sum("hiv_regnum_firstdiagnosis").alias("total_firstdiag"),
        F.round(F.avg("per_infectious_inalldoctors"), 2).alias("avg_share_inf")
    )
)

final.write.mode("overwrite").parquet(str(OUT / "hiv_by_year_parquet"))
final.coalesce(1).write.mode("overwrite").option("header", True).csv(str(OUT / "hiv_by_year_csv"))

print("Строк в final:", final.count())

Строк в final: 8


Итоговый DataFrame по годам содержит всего 8 строк — это очень маленький результат. Поэтому его можно записать в CSV одним файлом через coalesce(1): для такого объёма это удобно, потому что не создаётся лишних part-файлов. Для больших таблиц так делать нельзя — там coalesce(1) сведёт всю запись в один поток и замедлит работу.

In [31]:
back = spark.read.parquet(str(OUT / "hiv_by_year_parquet"))

print("Строк до записи:", final.count())
print("Строк после чтения:", back.count())

print("Схема до записи:")
final.printSchema()

print("Схема после чтения:")
back.printSchema()

Строк до записи: 8
Строк после чтения: 8
Схема до записи:
root
 |-- year: integer (nullable = true)
 |-- n_territories: long (nullable = false)
 |-- avg_firstdiag_100k: double (nullable = true)
 |-- total_firstdiag: double (nullable = true)
 |-- avg_share_inf: double (nullable = true)

Схема после чтения:
root
 |-- year: integer (nullable = true)
 |-- n_territories: long (nullable = true)
 |-- avg_firstdiag_100k: double (nullable = true)
 |-- total_firstdiag: double (nullable = true)
 |-- avg_share_inf: double (nullable = true)



Parquet после round-trip сохранил те же 8 строк и ту же схему (integer, long, double). Значит, витрина пригодна для дальнейшей аналитики без ручного восстановления типов.

In [32]:
spark.stop()

### **Выводы по блоку 2**

1. Эпидемия по числу новых диагнозов снижается. Средний первичный диагноз на 100 тыс. населения вырос с 40,89 в 2012 до пика 55,84 в 2015, а затем опустился до 49,41 в 2019. Общее число впервые выявленных случаев сократилось с 300 444 в 2015 до 238 681 в 2019.

2. Проблема распределена по стране очень неравномерно. В 2019 году максимум — у Кемеровской области (164,3 на 100 тыс.), что более чем в 3 раза выше среднего по стране. В топ также входят Пермский край, Иркутская область, Новосибирская область и Свердловская область.

3. Положительных ИФА среди россиян больше, чем среди иностранцев. Доля положительных среди российских тестов — около 0,4 %, среди иностранных — около 0,1–0,2 %. Разница объясняется тем, что иностранцы чаще проходят тест в рамках обязательных процедур (например, при трудоустройстве), а не по эпидемиологическим показаниям.

4. Нагрузка на инфекционистов высокая и стабильная. В среднем на одного инфекциониста приходится около 12–15 первичных диагнозов за год. За 2012–2019 годы этот показатель не снизился — значит, рост числа врачей примерно соответствует динамике выявляемости.

5. Пропуски в тестовых данных — это разрыв методологии, а не ошибка. За 2016–2018 годы данные о тестировании отсутствуют (295 строк из 760). Их нельзя ни удалять, ни заменять нулями: это означало бы либо потерю реальных наблюдений, либо искусственное искажение средних.

6. Данные хорошо агрегируются и пригодны для витрины. Итоговая таблица по годам содержит 8 строк и стабильно сохраняется в Parquet со схемой (integer, long, double). После повторного чтения число строк и схема совпадают — витрину можно использовать без ручной правки типов.
### **Ограничения по блоку 2**

1. Пропуски за 2016–2018 годы ограничивают выводы о динамике. Именно в эти годы отсутствуют данные о тестировании, поэтому полноценно оценить связь между числом тестов и числом выявленных случаев за весь период нельзя. Анализ по годам фактически опирается на 2012–2015 и 2019 годы.

2. Регионы сильно различаются по размеру. Проценты и средние по 95 территориям нельзя интерпретировать без учёта численности населения. Высокий показатель на 100 тыс. в Кемеровской области не обязательно означает, что там ситуация хуже, чем в крупных агломерациях — это следствие разного масштаба территорий и разной практики тестирования.